In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://staging.rdm.example.com/'
idp_name_1 = 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
s3_access_key_1 = None
s3_secret_access_key_1 = None
s3_default_region_1 = None
s3_test_bucket_name_1 = None
s3_access_key_2 = None
s3_secret_access_key_2 = None
s3_default_region_2 = None
s3_test_bucket_name_2 = None
enable_52gb_file_upload = False
enable_1gb_file_upload = True
skip_130mb_upload = False
default_result_path = None
close_on_fail = False
transition_timeout = 10000
s3compat_type_name_1 = None
s3compat_type_name_2 = None
skip_failed_test = True
skip_preview_check = False
skip_too_many_files_check = False
exclude_notebooks = []

target_storage_name = 'Amazon S3'
target_storage_id = 's3'
rdm_project_prefix = 'TEST-{}-{}'.format(target_storage_id.upper(), datetime.now().strftime('%Y%m%d-%H%M%S'))

In [ ]:
# # rdm_url = 'https://bh.rdm.yzwlab.com/'
# # idp_name_1 = None
# target_storage_name = 'S3 Compatible Storage'
# target_storage_id = 's3compat'
# s3compat_type_name_1 = 'Wasabi'
# s3compat_type_name_2 = 'SAKURA Cloud'

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
if s3_access_key_1 is None:
    s3_access_key_1 = input(prompt=f'{target_storage_name} Access Key #1')
if s3_secret_access_key_1 is None:
    s3_secret_access_key_1 = getpass(prompt=f'{target_storage_name} Secret Access Key #1')
if s3_default_region_1 is None:
    s3_default_region_1 = input(prompt=f'{target_storage_name} Region #1')
if s3_test_bucket_name_1 is None:
    s3_test_bucket_name_1 = input(f'{target_storage_name} Bucket Name #1')
(s3_access_key_1, s3_default_region_1, s3_test_bucket_name_1)

In [ ]:
# S3認証キー #2 は #1 と異なるアカウントのものである必要あり
if s3_access_key_2 is None:
    s3_access_key_2 = input(prompt=f'{target_storage_name} Access Key #2')
if s3_secret_access_key_2 is None:
    s3_secret_access_key_2 = getpass(prompt=f'{target_storage_name} Secret Access Key #2')
if s3_default_region_2 is None:
    s3_default_region_2 = input(prompt=f'{target_storage_name} Region #2')
if s3_test_bucket_name_2 is None:
    s3_test_bucket_name_2 = input(f'{target_storage_name} Bucket Name #2')
(s3_access_key_2, s3_default_region_2, s3_test_bucket_name_2)

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# GakuNinRDM 総合テスト [S3ストレージ]

- サブシステム名: ログイン
- ページ/アドオン: トップページ
- 機能分類: ストレージ制御確認
- シナリオ名: *
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
(target_storage_id, target_storage_name)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「設定」を選択する


In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[@data-analytics-name="Settings"]').click()

    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンアカウント構成」を選択する


In [ ]:
import traceback

has_connections = False

async def _step(page):
    global has_connections
    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[text() = "{target_storage_name}"]')).to_be_visible(timeout=transition_timeout)
    # 30秒以内に「アカウントを切断」が1つ以上表示されるかを確認
    try:
        await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=30000)
        has_connections = True
    except:
        print('Ignored')
        traceback.print_exc()
        has_connections = False

await run_pw(_step)

## (「アカウントを切断」が対象ストレージに表示されている場合)すべての「アカウントを切断」をクリックし、「Disconnect」をクリックする



In [ ]:
async def _step(page):
    if not has_connections:
        return
    try:
        await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=30000)
    except:
        print('Ignored')
        traceback.print_exc()
        return
    while True:
        print('Disconnecting...')
        await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]').click()
        await page.locator('.bootbox-confirm .btn-danger').click()
        # 30秒以内に「アカウントを切断」が表示されれば繰り返し
        try:
            # 前のものが消えるまで待つ...
            time.sleep(3)
            await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=30000)
        except:
            print('Ignored')
            traceback.print_exc()
            break
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 左上 GakuNin RDMロゴをクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "service-name")]//a').click()
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TEST-対象ストレージ-YYYYMMDD-HHMMSS-dashboard」プロジェクトを作成する

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, f'{rdm_project_prefix}-dashboard', transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_prefix}-dashboard"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, 'NII Storage')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

In [ ]:
addon_name = target_storage_name

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await expect(page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内対象ストレージの行の「有効にする」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]').click()
    
    await expect(page.locator(f'//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[@data-bb-handler = "confirm"]').click()
    
    await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "アカウントに接続する")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アカウントに接続する」リンクをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "アカウントに接続する")]').click()

    await expect(page.locator(f'#{target_storage_id}InputCredentials .btn-success')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## アクセスキーとシークレットキー(1つ目)を入力する

In [ ]:
async def _step(page):
    if target_storage_id == 's3compat':
        await page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//select[@id = "selected_service"]').select_option(s3compat_type_name_1)
        access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@name = "access_key"]')
        secret_access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@name = "secret_key"]')
    else:
        access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@id = "access_key"]')
        secret_access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@id = "secret_key"]')
    await access_key_field.fill(s3_access_key_1)
    await secret_access_key_field.fill(s3_secret_access_key_1)

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'#{target_storage_id}InputCredentials .btn-success').click()

    await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「接続」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "接続"]').click()

    await expect(page.locator(f'//button[@id = "newBucket"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 指定されたバケットを選択する

In [ ]:
async def _step(page):
    await page.locator(f'//span[text() = "{s3_test_bucket_name_1}"]/../../..//input[@type = "radio"]').click()

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
import asyncio

async def _step(page):
    # ここでdelayを入れないと 正常にリンクされました メッセージが表示されない(設定自体はなされるよう)
    await asyncio.sleep(1)
    await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=30000)

await run_pw(_step)

## 「ファイルページ」リンクをクリックする

In [ ]:
# import scripts.grdm
# importlib.reload(scripts.grdm)
import traceback

async def _step(page):
    try:
        await page.get_by_role("link", name="ファイルページ").click()
    except:
        # GitHub Actions環境では、前のセルで「正常にリンクされました」が出ても、この段階でファイルページリンクが消えることがある
        # そのため、ファイルタブをクリックすることでフォールバックとする
        print('Ignored')
        traceback.print_exc()
        await page.locator('//*[@id = "projectNavFiles"]//a').click()


    # ストレージが表示されること
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## プロジェクトダッシュボードでの「ファイル基本操作」テストの実施

テスト「テスト手順-ストレージ共通-ファイル基本操作」をプロジェクトダッシュボードで実施する。

In [ ]:
from datetime import datetime
import os
import papermill as pm
import traceback
from scripts.papermillHelpers import gen_run_notebook

def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir

result_dir = make_result_dir(default_result_path)

run_notebook = gen_run_notebook(
    result_dir,
    transition_timeout,
    dict(
        rdm_url=rdm_url,
        idp_name_1=idp_name_1,
        idp_username_1=idp_username_1,
        idp_password_1=idp_password_1,
    ),
    skip_failed_test,
    exclude_notebooks,
)

result_notebooks = []
result_dir

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-ストレージ共通-ファイル基本操作.ipynb',
    dict(
        enable_52gb_file_upload=False,
        target_storage_name=target_storage_name,
        target_file_view='project-dashboard',
        rdm_project_name=f'{rdm_project_prefix}-dashboard',
        enable_1gb_file_upload=enable_1gb_file_upload,
        skip_preview_check=skip_preview_check,
        skip_130mb_upload=skip_130mb_upload,
    ),
    f'-ファイル基本操作-プロジェクトダッシュボード-{target_storage_name}',
))
result_notebooks[-1]

## 左上 GakuNin RDMロゴをクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "service-name")]//a').click()
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TEST-対象ストレージ-YYYYMMDD-HHMMSS-filetab」プロジェクトを作成する

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, f'{rdm_project_prefix}-filetab', transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_prefix}-filetab"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, 'NII Storage')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

In [ ]:
addon_name = target_storage_name

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await expect(page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内対象ストレージの行の「有効にする」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]').click()
    
    await expect(page.locator(f'//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[@data-bb-handler = "confirm"]').click()
    
    await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プロフィールからアカウントをインポート」リンクをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]').click()

    await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「接続」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "接続"]').click()

    await expect(page.locator(f'//button[@id = "newBucket"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 指定されたバケットを選択する

In [ ]:
async def _step(page):
    await page.locator(f'//span[text() = "{s3_test_bucket_name_1}"]/../../..//input[@type = "radio"]').click()

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
async def _step(page):
    # ここでdelayを入れないと 正常にリンクされました メッセージが表示されない(設定自体はなされるよう)
    time.sleep(1)
    await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=30000)

await run_pw(_step)

## 「ファイルページ」リンクをクリックする

In [ ]:
# import scripts.grdm
# importlib.reload(scripts.grdm)

async def _step(page):
    try:
        await page.get_by_role("link", name="ファイルページ").click()
    except:
        # GitHub Actions環境では、前のセルで「正常にリンクされました」が出ても、この段階でファイルページリンクが消えることがある
        # そのため、ファイルタブをクリックすることでフォールバックとする
        print('Ignored')
        traceback.print_exc()
        await page.locator('//*[@id = "projectNavFiles"]//a').click()

    # ストレージが表示されること
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## ファイルタブでの「ファイル基本操作」テストの実施

テスト「テスト手順-ストレージ共通-ファイル基本操作」をファイルタブで実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-ストレージ共通-ファイル基本操作.ipynb',
    dict(
        enable_52gb_file_upload=False,
        target_storage_name=target_storage_name,
        target_file_view='file-tab',
        rdm_project_name=f'{rdm_project_prefix}-filetab',
        enable_1gb_file_upload=enable_1gb_file_upload,
        skip_preview_check=skip_preview_check,
        skip_130mb_upload=skip_130mb_upload,
    ),
    f'-ファイル基本操作-ファイルタブ-{target_storage_name}',
))
result_notebooks[-1]

## 左上 GakuNin RDMロゴをクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "service-name")]//a').click()
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TEST-対象ストレージ-YYYYMMDD-HHMMSS-metadata」プロジェクトを作成する

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, f'{rdm_project_prefix}-metadata', transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_prefix}-metadata"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, 'NII Storage')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

In [ ]:
addon_name = target_storage_name

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await expect(page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内対象ストレージの行の「有効にする」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]').click()
    
    await expect(page.locator(f'//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[@data-bb-handler = "confirm"]').click()
    
    await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プロフィールからアカウントをインポート」リンクをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]').click()

    await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「接続」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "接続"]').click()

    await expect(page.locator(f'//button[@id = "newBucket"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 指定されたバケットを選択する

In [ ]:
async def _step(page):
    await page.locator(f'//span[text() = "{s3_test_bucket_name_1}"]/../../..//input[@type = "radio"]').click()

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
async def _step(page):
    # ここでdelayを入れないと 正常にリンクされました メッセージが表示されない(設定自体はなされるよう)
    time.sleep(1)
    await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=30000)

await run_pw(_step)

## 「ファイルページ」リンクをクリックする

In [ ]:
# import scripts.grdm
# importlib.reload(scripts.grdm)

async def _step(page):
    try:
        await page.get_by_role("link", name="ファイルページ").click()
    except:
        # GitHub Actions環境では、前のセルで「正常にリンクされました」が出ても、この段階でファイルページリンクが消えることがある
        # そのため、ファイルタブをクリックすることでフォールバックとする
        print('Ignored')
        traceback.print_exc()
        await page.locator('//*[@id = "projectNavFiles"]//a').click()

    # ストレージが表示されること
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## ストレージでの「Metadataアドオンの登録」テストの実施

テスト「テスト手順-ストレージ共通-Metadataアドオン」をファイルタブで実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-ストレージ共通-Metadataアドオン.ipynb',
    dict(
        target_storage_name=target_storage_name,
        target_storage_id=target_storage_id,
        rdm_project_name=f'{rdm_project_prefix}-metadata'
    ),
    f'Metadataアドオン-{target_storage_name}',
))
result_notebooks[-1]

## ユーザーメニューから「設定」を選択する


In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[@href="/settings/"]').click()

    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンアカウント構成」を選択する


In [ ]:
import traceback

async def _step(page):
    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[@data-bind="text: properName" and text() = "{target_storage_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージの「アカウントの接続または再認証」をクリックする


In [ ]:
async def _step(page):
    await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/..//*[text() = "アカウントの接続または再認証"]').click()

    await expect(page.locator(f'#{target_storage_id}InputCredentials .btn-success')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## アクセスキーとシークレットキー(2つ目)を入力する

In [ ]:
async def _step(page):
    if target_storage_id == 's3compat':
        await page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//select[@id = "selected_service"]').select_option(s3compat_type_name_2)
        access_key_field = page.locator(f'//*[@id = "s3compatInputCredentials"]//input[@name = "access_key"]')
        secret_access_key_field = page.locator(f'//*[@id = "s3compatInputCredentials"]//input[@name = "secret_key"]')
    else:
        access_key_field = page.locator(f'//input[@id = "access_key"]')
        secret_access_key_field = page.locator(f'//input[@id = "secret_key"]')
    await access_key_field.fill(s3_access_key_2)
    await secret_access_key_field.fill(s3_secret_access_key_2)

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'#{target_storage_id}InputCredentials .btn-success').click()

    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(2, timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 左上 GakuNin RDMロゴをクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "service-name")]//a').click()
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TEST-対象ストレージ-YYYYMMDD-HHMMSS-toomany」プロジェクトを作成する

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, f'{rdm_project_prefix}-toomany', transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_prefix}-toomany"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, 'NII Storage')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

In [ ]:
addon_name = target_storage_name

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await expect(page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内対象ストレージの行の「有効にする」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//div[@full_name = "{addon_name}"]//descendant::a[text() = "有効にする"]').click()
    
    await expect(page.locator(f'//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[@data-bb-handler = "confirm"]').click()
    
    await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プロフィールからアカウントをインポート」リンクをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "プロフィールからアカウントをインポート")]').click()

    await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## アカウントドロップダウンの2つ目を選択する

In [ ]:
async def _step(page):
    await page.locator('//select[contains(@class, "bootbox-input-select")]').select_option(index=1)

    await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「接続」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "接続"]').click()

    await expect(page.locator(f'//button[@id = "newBucket"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 指定されたバケットを選択する

In [ ]:
async def _step(page):
    await page.locator(f'//span[text() = "{s3_test_bucket_name_2}"]/../../..//input[@type = "radio"]').click()

await run_pw(_step)

## 「保存」をクリックする

In [ ]:
async def _step(page):
    # ここでdelayを入れないと 正常にリンクされました メッセージが表示されない(設定自体はなされるよう)
    time.sleep(1)
    await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=30000)

await run_pw(_step)

## (実施対象の場合)テストバケット(2つ目)に1500個のファイルを作る

ファイル名は `NNNN` の形式とする。

In [ ]:
# with open(os.path.join(work_dir, '.aws'), 'w') as f:
#     f.write(f'''
# export AWS_ACCESS_KEY_ID={s3_access_key_2}
# export AWS_SECRET_ACCESS_KEY={s3_secret_access_key_2}
# export AWS_DEFAULT_REGION={s3_default_region_2}
# ''')
# # TODO
# !source {work_dir}/.aws && ~/aws/dist/aws s3api list-objects --bucket {s3_test_bucket_name_2}

In [ ]:
# for i in range(1500):
#     filename = '{0:04d}'.format(i + 1)
#     !source {work_dir}/.aws && ~/aws/dist/aws s3api put-object --bucket {s3_test_bucket_name_2} --key {filename}

## 「ファイルページ」リンクをクリックする

In [ ]:
# import scripts.grdm
# importlib.reload(scripts.grdm)

async def _step(page):
    try:
        await page.get_by_role("link", name="ファイルページ").click()
    except:
        # GitHub Actions環境では、前のセルで「正常にリンクされました」が出ても、この段階でファイルページリンクが消えることがある
        # そのため、ファイルタブをクリックすることでフォールバックとする
        print('Ignored')
        traceback.print_exc()
        await page.locator('//*[@id = "projectNavFiles"]//a').click()

    # ストレージが表示されること
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## (実施対象の場合)ファイルが 0001 〜 1500 まであることを確認する

999個よりあとは、スクロールによって読み込まれること

In [ ]:
async def _step(page):
    if skip_too_many_files_check:
        return
    #await page.reload()
    for i in range(1500):
        filename = '{0:04d}'.format(i + 1)
        await grdm.get_select_file_extension_locator(page, filename).scroll_into_view_if_needed()
        await page.evaluate("document.querySelector('#tb-tbody').scrollBy(0, 40)")

await run_pw(_step)

## プロジェクトを削除する

In [ ]:
async def _step(page):
    await scripts.grdm.delete_project(page)
    
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「設定」を選択する


In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[@data-analytics-name="Settings"]').click()

    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンアカウント構成」を選択する


In [ ]:
import traceback

async def _step(page):
    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[@data-bind="text: properName" and text() = "{target_storage_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(2, timeout=30000)

await run_pw(_step)

## 1つ目の「アカウントを切断」をクリックし、「Disconnect」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]').click()
    await page.locator('.bootbox-confirm .btn-danger').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 2つ目の「アカウントを切断」をクリックし、「Disconnect」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]').click()
    await page.locator('.bootbox-confirm .btn-danger').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}